<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/_banners/opim5641_banner.svg" width="100%" alt="OPIM 5641 banner"/>

**Dr. Dave Wanik - Operations and Information Management - University of Connecticut**

---

# Studio 4.3 — Real Stock Data, Done Right (and How Did You Do?)

In 4.2 the data arrived clean: a table of monthly returns, ready for the model. Real data does not arrive that way.
Tonight you each take one **sector** of the stock market (five large companies) and
does what an analyst would do:

1. **Part 1 · Prices are not returns.** Turn month-end prices into monthly returns.
2. **Part 2 · The split trap.** Why you always use *adjusted* prices.
3. **Part 3 · Check the data before you trust it.**
4. **Part 4 · Your sector, using only the past.** Build the efficient frontier on 48 months and pick a portfolio.
5. **Part 5 · How did you do?** Hold that portfolio through 12 months the model never saw, against the S&P 500.

All of the code is written. Read, predict (write your guess down first), then run.

## Import modules (download Pyomo and the ipopt solver)

Run this first, every time. It installs Pyomo and **ipopt**, the nonlinear solver we use in Module 4.
It takes about a minute.

In [ ]:
# install Pyomo and the ipopt solver in Colab (idaes-pse brings both)
import sys
import os

if 'google.colab' in sys.modules:
    !pip install -q idaes-pse --pre
    !idaes get-extensions --to ./bin
    os.environ['PATH'] += ':bin'

# this makes plots using pylab (matplotlib)
%matplotlib inline
from pylab import *
import numpy as np
import pandas as pd

# import ALL NAMESPACES from pyomo
from pyomo.environ import *

# where the setup cell put the solver
ipopt_path = '/content/bin/ipopt'

# Part 1 · Prices are not returns

The file below holds the **month-end price** of 55 stocks (11 sectors, 5 companies each), the S&P 500 fund (SPY), and
one fund per sector, from September 2021 to September 2026. The prices are already **adjusted** for stock splits and
dividends (more on that in Part 2).

In [ ]:
# month-end prices: one row per month, one column per ticker
data_folder = 'https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/studio4/data/'
prices = pd.read_csv(data_folder + 'sector_prices_monthly.csv', index_col=0)
lookup = pd.read_csv(data_folder + 'sector_lookup.csv')

print(prices.shape[0], "months,", prices.shape[1], "tickers")
prices[['AAPL', 'COST', 'NVDA', 'SPY']].head()

**Predict first.** Below are the prices of Apple, Costco and NVIDIA. Before you run it: which one do you
think made its owners the most money over these five years?

In [ ]:
# the raw prices
for ticker in ['AAPL', 'COST', 'NVDA']:
    plot(prices.index, prices[ticker], label=ticker)
xticks(prices.index[::12])
ylabel('Price ($)')
title('Month-end prices')
legend()
show()

Hard to say: Costco costs several hundred dollars a share and NVIDIA started near \$20, so the picture mostly
shows which share is **expensive**, not which one **grew**. The fix is to put every stock on the same scale: what
happened to **\$100** invested in each one at the start?

In [ ]:
# growth of $100: divide every price by its first price
for ticker in ['AAPL', 'COST', 'NVDA']:
    growth = 100 * prices[ticker] / prices[ticker].iloc[0]
    plot(prices.index, growth, label=ticker)
xticks(prices.index[::12])
ylabel('Value of $100 invested in Sep 2021')
title('Same scale: growth of $100')
legend()
show()

### From prices to monthly returns

The model never looks at prices. It needs the **monthly return**: the percent change from one month-end to the next,

$$r_t = \frac{P_t - P_{t-1}}{P_{t-1}} = \frac{P_t}{P_{t-1}} - 1$$

**Your turn, on paper.** Here are Costco's first two month-end prices. Work out the return for October 2021, then type
it in the cell below as a decimal (for example, 0.05 for 5%).

In [ ]:
print("Costco, Sep 2021: $%.2f" % prices.loc['2021-09', 'COST'])
print("Costco, Oct 2021: $%.2f" % prices.loc['2021-10', 'COST'])

In [ ]:
# type your answer here, then run the cell
my_answer = 0.0

actual = prices.loc['2021-10', 'COST'] / prices.loc['2021-09', 'COST'] - 1
if abs(my_answer - actual) < 0.0005:
    print("Correct: %.4f, or %.2f%%" % (actual, 100 * actual))
else:
    print("Not quite. Divide October's price by September's, then subtract 1.")

pandas does that for every stock and every month at once with `pct_change()`. The first row comes back empty
(September 2021 has no month before it), so we drop it: **61 prices give 60 returns.**

In [ ]:
returns = prices.pct_change()
print("first row (no month before it):")
print(returns.iloc[0, :4])

returns = returns.dropna()
print()
print(len(returns), "monthly returns, from", returns.index[0], "to", returns.index[-1])
returns[['AAPL', 'COST', 'NVDA', 'SPY']].head()

# Part 2 · The split trap

On June 10, 2024, NVIDIA did a **10-for-1 stock split**: every share became ten shares, each worth one tenth as much.
Nobody lost a dollar. But if you download the price **as it was quoted** on each day, here is what you get:

In [ ]:
nvda = pd.read_csv(data_folder + 'nvda_split_demo.csv', index_col=0)
nvda

**Predict first.** Using the *as-quoted* prices, what monthly return will the computer report for June 2024?
Write down your guess, then run the cell.

In [ ]:
nvda_returns = nvda.pct_change().dropna()

plot(nvda_returns.index, 100 * nvda_returns['close_as_quoted'], marker='o', label='as quoted')
plot(nvda_returns.index, 100 * nvda_returns['close_adjusted'], marker='o', label='adjusted')
axhline(0, color='gray', linewidth=0.8)
xticks(rotation=45)
ylabel('Monthly return (%)')
title('NVIDIA: did it really crash in June 2024?')
legend()
show()

print("June 2024 return, as quoted: %.1f%%" % (100 * nvda_returns.loc['2024-06', 'close_as_quoted']))
print("June 2024 return, adjusted:  %.1f%%" % (100 * nvda_returns.loc['2024-06', 'close_adjusted']))

One bad row is enough to ruin the model. Look at what it does to the two numbers the optimizer runs on:

In [ ]:
print("                         as quoted    adjusted")
print("average monthly return:   %8.4f    %8.4f" % (nvda_returns['close_as_quoted'].mean(), nvda_returns['close_adjusted'].mean()))
print("variance (the risk):      %8.4f    %8.4f" % (nvda_returns['close_as_quoted'].var(), nvda_returns['close_adjusted'].var()))

With the as-quoted prices, one of the best stocks of the decade looks like an ordinary stock (about 1.6% a month
instead of 8.4%) with more than **five times** the risk, and the optimizer would steer you away from it. **Always use adjusted prices.** With `yfinance`,
`auto_adjust=True` does this for you, and the file we loaded in Part 1 is already adjusted.

# Part 3 · Check the data before you trust it

Three quick checks, every time you get new data:

1. **How many months, from when to when?** (Enough history? The same months for every stock?)
2. **Any missing values?** (A company that did not exist yet, a ticker that changed.)
3. **Any impossible moves?** (A -90% month is usually a split or a typo, not a crash.)

In [ ]:
# 1. how many months, from when to when
print("months:", len(returns), "from", returns.index[0], "to", returns.index[-1])

# 2. missing values (every stock should have all 60 months)
print("missing values:", returns.isna().sum().sum())

# 3. the biggest one-month moves (are any of them suspicious?)
moves = pd.DataFrame({'worst month': returns.min(), 'best month': returns.max()})
moves.sort_values('worst month').head(8).round(3)

**Think about it.** The biggest moves above are all real (look them up if you are curious). None of
them is -90% or worse. What would you do if one of your stocks had a month like that?

# Part 4 · Your sector, using only the past

Here is the rule that makes this a fair test. In real life you only know the past. So we split the 60 months in two:

| | months | what we use it for |
|---|---|---|
| **The past** | October 2021 to September 2025 (48 months) | build the model and choose a portfolio |
| **The future** | October 2025 to September 2026 (12 months) | hidden from the model; Part 5 checks how you did |

**Pick your sector** from the menu, then click on the cell and choose **Runtime → Run cell and below**
(Ctrl+F10).

In [ ]:
#@title Pick your sector
sector = "Technology"  #@param ["Technology", "Financials", "Health Care", "Energy", "Consumer Staples", "Consumer Discretionary", "Industrials", "Utilities", "Communication Services", "Materials", "Real Estate"]

my_stocks = []
for i in range(len(lookup)):
    if lookup.loc[i, 'sector'] == sector:
        my_stocks.append(lookup.loc[i, 'ticker'])
        sector_fund = lookup.loc[i, 'sector_etf']

past = returns.loc[:'2025-09', my_stocks]
future = returns.loc['2025-10':, my_stocks]

# the two numbers the model needs, from the PAST only
df_return = past.mean()
df_cov = past.cov()

print("Sector:", sector, "  (sector fund:", sector_fund + ")")
print("The past:  ", len(past), "months,", past.index[0], "to", past.index[-1])
print("The future:", len(future), "months,", future.index[0], "to", future.index[-1], "(hidden from the model)")
companies = lookup.set_index('ticker').loc[my_stocks, ['company']]
companies['average monthly return'] = df_return
companies['volatility (std. dev.)'] = past.std()
companies.round(4)

**Predict first.** Look at the table. If you could only pick **one** of the five, which would it be? Write it
down. We will see what the model does with all five.

### The model (the same one as 4.2)

$$
\begin{aligned}
\max \quad & \sum_k \bar r_k \, x_k && \text{(average return)} \\
\text{subject to} \quad & \sum_k x_k = 1 && \text{(invest all of the money)} \\
& \sum_i \sum_j x_i \, x_j \, \text{Cov}(i,j) \le \text{risk limit} && \text{(the weighted covariance matrix, summed)} \\
& 0 \le x_k \le 1
\end{aligned}
$$

One change from 4.2: before drawing the frontier we solve for the **safest possible mix** (the smallest risk any
portfolio of these five can have), so the frontier starts exactly where a portfolio first becomes possible. That
works for every sector, even one where the stocks are all alike.

In [ ]:
import logging
logging.getLogger('pyomo.core').setLevel(logging.ERROR)


def build_model(stocks):
    model = ConcreteModel()
    model.x = Var(stocks, domain=NonNegativeReals, bounds=(0, 1))
    for s in stocks:
        model.x[s] = 1 / len(stocks)
    model.constraints = ConstraintList()
    budget = 0
    for s in stocks:
        budget += model.x[s]
    model.constraints.add(budget == 1)
    return model


def risk_of(model, stocks):
    risk_expr = 0
    for i in stocks:
        for j in stocks:
            risk_expr += model.x[i] * df_cov.loc[i, j] * model.x[j]
    return risk_expr


def safest_mix():
    # minimize the risk, nothing else: the far left end of the frontier
    stocks = list(df_return.index)
    model = build_model(stocks)
    model.risk = Objective(expr=risk_of(model, stocks), sense=minimize)
    SolverFactory('ipopt', executable=ipopt_path).solve(model)
    return model.risk()


def solve_portfolio(risk_limit):
    # the 4.2 model: best average return with the risk under the limit
    stocks = list(df_return.index)
    model = build_model(stocks)
    obj_expr = 0
    for s in stocks:
        obj_expr += df_return[s] * model.x[s]
    model.portfolio_return = Objective(expr=obj_expr, sense=maximize)
    risk_expr = risk_of(model, stocks)
    model.constraints.add(risk_expr <= risk_limit)
    results = SolverFactory('ipopt', executable=ipopt_path).solve(model)
    if results.solver.termination_condition != TerminationCondition.optimal:
        return None
    weights = {}
    for s in stocks:
        weights[s] = model.x[s]()
    return {'return': model.portfolio_return(), 'risk': value(risk_expr), 'weights': weights}


# the frontier: from the safest mix up to the risk of the best-returning stock
bottom = safest_mix() * 1.0001
best_stock = df_return.idxmax()
top = df_cov.loc[best_stock, best_stock]
param_analysis = {}
frontier_returns = {}
for r in np.linspace(bottom, top, 30):
    answer = solve_portfolio(r)
    if answer is None:
        continue
    param_analysis[r] = answer['weights']
    frontier_returns[r] = answer['return']
param_analysis = pd.DataFrame(param_analysis).T
frontier_returns = pd.Series(frontier_returns)
print(len(frontier_returns), "portfolios on the frontier, risk from %.5f to %.5f" % (bottom, top))

In [ ]:
# the frontier and what to hold along it, built from the past only
fig, axes = subplots(1, 2, figsize=(14, 5))
axes[0].plot(frontier_returns.index, frontier_returns.values, marker='o', markersize=4, label='efficient frontier')
for s in my_stocks:
    axes[0].scatter(df_cov.loc[s, s], df_return[s], color='gray')
    axes[0].annotate(s, (df_cov.loc[s, s], df_return[s]), fontsize=9)
axes[0].set_xlabel('Risk (variance of the monthly return)')
axes[0].set_ylabel('Average monthly return')
axes[0].set_title(sector + ': the efficient frontier (the past)')
axes[0].legend(fontsize=8)

axes[1].stackplot(param_analysis.index, param_analysis.T.values, labels=param_analysis.columns)
axes[1].set_xlabel('Risk (variance of the monthly return)')
axes[1].set_ylabel('Share of the money')
axes[1].set_title('What to hold along the frontier')
axes[1].legend(loc='center left', bbox_to_anchor=(1, 0.5), fontsize=8)
tight_layout()
show()

### Choose your portfolio

Pick a risk tolerance: 0 is the safest mix, 1 is all-in on the best stock of the past. Run the
cell and **write down your weights** before you look at the future.

In [ ]:
#@title Your risk tolerance
risk_tolerance = 0.3  #@param {type:"slider", min:0, max:1, step:0.05}

my_limit = frontier_returns.index.min() + risk_tolerance * (frontier_returns.index.max() - frontier_returns.index.min())
mine = solve_portfolio(my_limit)
my_weights = pd.Series(mine['weights'])

safest = frontier_returns.index.min()
riskiest = frontier_returns.index.max()
print("risk tolerance %.2f = %.0f%% of the way from the safest mix (risk %.4f) to all-in on %s (risk %.4f)"
      % (risk_tolerance, 100 * risk_tolerance, safest, df_return.idxmax(), riskiest))
print("risk limit = %.4f + %.2f x (%.4f - %.4f) = %.4f" % (safest, risk_tolerance, riskiest, safest, my_limit))
print("-> expected %.2f%% a month, with monthly volatility of %.2f%%" % (100 * mine['return'], 100 * np.sqrt(mine['risk'])))
print()
print("Your portfolio:")
for s in my_weights.sort_values(ascending=False).index:
    if my_weights[s] > 0.005:
        print("  %-6s %5.1f%%" % (s, 100 * my_weights[s]))

# Part 5 · How did you do?

Now we open the envelope. We invest **\$10,000** in your portfolio on October 1, 2025, and keep the same weights every
month (we rebalance back to them at each month-end) through September 2026. Each month the portfolio earns the
weighted average of its stocks' returns,

$$r^{\text{portfolio}}_t = \sum_k x_k \, r_{k,t}, \qquad \text{value}_T = 10{,}000 \times \prod_{t=1}^{T} \left(1 + r^{\text{portfolio}}_t\right)$$

and we compare it with three simple alternatives: an even split of your five stocks, your sector's fund, and the
S&P 500 (SPY).

In [ ]:
# each month's return for your portfolio and for an even split of the same five stocks
my_month = 0
even_month = 0
for s in my_stocks:
    my_month = my_month + my_weights[s] * future[s]
    even_month = even_month + (1 / len(my_stocks)) * future[s]

contenders = {
    'Your portfolio': my_month,
    'Even split of your 5': even_month,
    'Sector fund (' + sector_fund + ')': returns.loc['2025-10':, sector_fund],
    'S&P 500 (SPY)': returns.loc['2025-10':, 'SPY'],
}

final = {}
for name in contenders:
    value_path = 10000 * (1 + contenders[name]).cumprod()
    plot(value_path.index, value_path.values, marker='o', markersize=3, label=name)
    final[name] = value_path.iloc[-1]
axhline(10000, color='gray', linewidth=0.8)
xticks(rotation=45)
ylabel('Value of $10,000')
title(sector + ': the 12 months the model never saw')
legend(fontsize=8)
show()

for name in final:
    print("%-28s $%9s" % (name, "{:,.0f}".format(final[name])))

**Report to the class.** Write your name, sector, risk tolerance and final value on the board. Who finished
highest? Who beat the S&P 500?

### Did the risk turn out the way the model expected?

The model promised a certain monthly volatility (from the past). Here is what actually happened in the future:

In [ ]:
print("expected monthly return:   %6.2f%%    actual: %6.2f%%" % (100 * mine['return'], 100 * my_month.mean()))
print("expected monthly volatility: %4.2f%%    actual: %6.2f%%" % (100 * np.sqrt(mine['risk']), 100 * my_month.std()))

### What real data cannot tell you

- **The past is not the future.** The averages come from 48 months. A stock that soared in the past can stall, and
  the optimizer, which loves the best past average, piles into it.
- **Survivorship.** We picked large, famous companies *today*. Companies that shrank or disappeared since 2021 are not
  in the file, so the past looks better than it really was.
- **Averages are noisy.** With 48 months, the average return of a single stock is very uncertain, and the model
  treats it as if it were known exactly.
- **Twelve months is one draw.** Whoever won tonight might lose next year with the same method.

### Your turn

Answer in a text cell:

1. In Part 1, why can't you compare stocks by their prices? What did the "growth of \$100" picture fix?
2. In Part 2, what did the split do to NVIDIA's average return and variance, and what would the optimizer have done
   with those numbers?
3. Which of your five stocks did you pick on your own in Part 4, and how much of the money did the model put in it?
4. How did your portfolio do against the even split, your sector fund and SPY? Was the actual volatility close to the
   expected one?
5. Re-run Parts 4 and 5 with risk tolerance 0 and with 1. Which did better in the future? Does that prove one of them
   is the better choice? Why or why not?

## Bottom line

- Real data arrives as **prices**. The model needs **returns**: $r_t = P_t / P_{t-1} - 1$.
- Use **adjusted** prices, or one split can wreck the average and the risk.
- **Check the data** (months, missing values, impossible moves) before you trust it.
- Build the model on **the past**, test it on a **future it never saw**, and compare against simple alternatives.
  That is how you find out whether a model is any good, and it is the backbone of your final project.